In [10]:
import pandas as pd
import numpy as np
import os

# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project"

BIN_LABELS = ["bin1_under_neg5", "bin2_neg5to0", "bin3_0to5",
              "bin4_5to10",      "bin5_10to15",  "bin6_15to20",
              "bin7_20to25",     "bin8_25to30",  "bin9_over30"]

# ── LOAD ─────────────────────────────────────────────────────────────────────
print("Loading data...")
sales_m   = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861m_sales.csv"))
territory = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861_territory.csv"))
fips_ref  = pd.read_csv(os.path.join(OUTPUT_DIR, "county_fips.csv"),
                        sep="|", dtype=str)
noaa      = pd.read_csv(os.path.join(OUTPUT_DIR, "noaa_panel_final.csv"),
                        dtype={"FIPS": str})
acs       = pd.read_csv(os.path.join(OUTPUT_DIR, "acs_poverty_panel.csv"),
                        dtype={"FIPS": str})

# Fix types
sales_m["year"]   = sales_m["year"].astype(int)
sales_m["month"]  = sales_m["month"].astype(int)
territory["year"] = territory["year"].astype(int)

print(f"Monthly sales: {len(sales_m)} rows")
print(f"Territory: {len(territory)} rows")
print(f"NOAA panel: {len(noaa)} rows")

# ── STEP 1: CLEAN COUNTY NAMES ────────────────────────────────────────────────
def clean_county(name):
    if pd.isna(name):
        return name
    s = str(name)
    s = pd.Series([s]).str.replace(
        r"\s+(County|Parish|Borough|Municipality|Census Area|"
        r"City and Borough|Municipio|District|Island|city|City)$",
        "", regex=True).iloc[0]
    s = s.strip().lower()
    s = s.replace("st. ", "saint ").replace("st ", "saint ")
    s = s.replace(".", "")
    return s.strip()

fips_ref.columns         = [c.strip().lower() for c in fips_ref.columns]
fips_ref["FIPS"]         = fips_ref["statefp"].str.zfill(2) + fips_ref["countyfp"].str.zfill(3)
fips_ref["county_clean"] = fips_ref["countyname"].apply(clean_county)
fips_ref["state_abbr"]   = fips_ref["state"].str.upper()

territory["county_clean"] = territory["county_name"].apply(clean_county)
territory["state_abbr"]   = territory["state"].str.upper()

# ── STEP 2: MERGE TERRITORY TO FIPS ──────────────────────────────────────────
territory_fips = territory.merge(
    fips_ref[["state_abbr", "county_clean", "FIPS"]],
    on=["state_abbr", "county_clean"], how="left"
)
print(f"Territory FIPS match rate: {territory_fips['FIPS'].notna().mean():.1%}")


# ── STEP 3: MERGE MONTHLY SALES TO TERRITORY ─────────────────────────────────
utility_county_m = sales_m.merge(
    territory_fips[["utility_id", "year", "state", "FIPS", "county_name"]],
    on=["utility_id", "year", "state"], how="inner"
)
print(f"Utility-county-month panel: {len(utility_county_m)} rows")
print(f"Unique FIPS: {utility_county_m['FIPS'].nunique()}")

# ── STEP 4: AGGREGATE TO COUNTY-MONTH ────────────────────────────────────────
county_month = (utility_county_m.dropna(subset=["FIPS"])
                .groupby(["FIPS", "year", "month"])
                .agg(
                    res_revenue_000_total = ("res_revenue_000", "sum"),
                    res_sales_mwh_total   = ("res_sales_mwh",   "sum"),
                    res_customers_total   = ("res_customers",   "sum"),
                    n_utilities           = ("utility_id",      "nunique"),
                )
                .reset_index())

# Implied price
county_month["implied_price_per_mwh"] = np.where(
    (county_month["res_revenue_000_total"] > 0) &
    (county_month["res_sales_mwh_total"]   > 0),
    (county_month["res_revenue_000_total"] * 1000) /
     county_month["res_sales_mwh_total"],
    np.nan
)

# Sales per customer
county_month["res_sales_per_customer"] = np.where(
    county_month["res_customers_total"] > 0,
    county_month["res_sales_mwh_total"] / county_month["res_customers_total"],
    np.nan
)

# Log transforms
county_month["log_res_sales_per_customer"] = np.log(
    county_month["res_sales_per_customer"].replace(0, np.nan))
county_month["log_implied_price"] = np.log(
    county_month["implied_price_per_mwh"].replace(0, np.nan))

# Dominant ownership
ownership_county_m = (utility_county_m
                      .dropna(subset=["FIPS", "res_customers"])
                      .sort_values("res_customers", ascending=False)
                      .groupby(["FIPS", "year", "month"])
                      .agg(dominant_ownership_type=("ownership_type", "first"))
                      .reset_index())

county_month = county_month.merge(
    ownership_county_m, on=["FIPS", "year", "month"], how="left")

print(f"\nCounty-month panel: {len(county_month)} rows")
print(f"Unique counties: {county_month['FIPS'].nunique()}")

# ── STEP 5: MERGE NOAA TAVG BINS AND PRCP ────────────────────────────────────
noaa["FIPS"]         = noaa["FIPS"].str.zfill(5)
acs["FIPS"]          = acs["FIPS"].str.zfill(5)
county_month["FIPS"] = county_month["FIPS"].str.zfill(5)

noaa_cols = ["FIPS", "year", "month"] + BIN_LABELS + ["prcp_monthly_mm"]

master_m = county_month.merge(
    noaa[noaa_cols],
    on=["FIPS", "year", "month"], how="left"
)

# ── STEP 6: MERGE ACS POVERTY (ANNUAL) ───────────────────────────────────────
master_m = master_m.merge(
    acs[["FIPS", "year", "pct_below_150pct_poverty", "total_pop"]],
    on=["FIPS", "year"], how="left"
)

# ── STEP 7: CONSTRUCT KEY VARIABLES ──────────────────────────────────────────
mean_pov = master_m["pct_below_150pct_poverty"].mean()
master_m["poverty_demeaned"] = master_m["pct_below_150pct_poverty"] - mean_pov

# Precipitation square (for regression control)
master_m["prcp_monthly_mm_sq"] = master_m["prcp_monthly_mm"] ** 2

# Heat x poverty interactions for the three hottest bins
for bin_label in ["bin7_20to25", "bin8_25to30", "bin9_over30"]:
    master_m[f"{bin_label}_x_poverty"] = (
        master_m[bin_label] * master_m["poverty_demeaned"]
    )

# County-month FE identifier
master_m["county_month_fe"] = (
    master_m["FIPS"] + "_" + master_m["month"].astype(str).str.zfill(2)
)

print(f"\nMaster monthly panel: {len(master_m)} rows")
print(f"Unique counties: {master_m['FIPS'].nunique()}")
print(f"Years: {sorted(master_m['year'].unique())}")

# Missing values
print(f"\nMissing values:")
key_vars = ["pct_below_150pct_poverty", "bin9_over30",
            "prcp_monthly_mm", "log_res_sales_per_customer",
            "log_implied_price"]
for v in key_vars:
    n = master_m[v].isna().sum()
    print(f"  {v}: {n} ({n/len(master_m):.1%})")

# Summary stats
print(f"\nSummary stats:")
print(master_m[["log_res_sales_per_customer", "log_implied_price",
                 "pct_below_150pct_poverty",
                 "bin8_25to30", "bin9_over30",
                 "prcp_monthly_mm"]].describe().round(3))

# Sanity check: Autauga summer 2020
autauga = master_m[(master_m["FIPS"] == "01001") &
                   (master_m["year"] == 2020) &
                   (master_m["month"].isin([6, 7, 8]))]
print(f"\nAutauga County 2020 summer months:")
print(autauga[["FIPS", "year", "month",
               "bin8_25to30", "bin9_over30", "prcp_monthly_mm",
               "log_res_sales_per_customer",
               "pct_below_150pct_poverty"]].to_string())

# ── SAVE ──────────────────────────────────────────────────────────────────────
master_m.to_csv(os.path.join(OUTPUT_DIR, "master_panel_monthly.csv"),
                index=False)
print(f"\nSaved master_panel_monthly.csv")
print(f"Final panel: {len(master_m)} county-month observations")

Loading data...
Monthly sales: 48455 rows
Territory: 118147 rows
NOAA panel: 372840 rows
Territory FIPS match rate: 99.5%
Utility-county-month panel: 512863 rows
Unique FIPS: 2814

County-month panel: 335878 rows
Unique counties: 2814

Master monthly panel: 335878 rows
Unique counties: 2814
Years: [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]

Missing values:
  pct_below_150pct_poverty: 300 (0.1%)
  bin9_over30: 2412 (0.7%)
  prcp_monthly_mm: 2412 (0.7%)
  log_res_sales_per_customer: 3 (0.0%)
  log_implied_price: 11 (0.0%)

Summary stats:
       log_res_sales_per_customer  log_implied_price  \
count                  335875.000         335867.000   
mean                       -0.109              4.842   
std                         0.324              0.245   
min                        -6.697              2.560   
25%                        -0.333              4.686   
50%                        -0.111              4.808   
75%                         0.129              4

In [5]:
import pandas as pd

fpath = "/Users/aliabonanno/public_policy_project/tempdata/prcp-202007-cty-scaled.csv"
day_cols = [f"day{i}" for i in range(1, 32)]
col_names = ["type", "FIPS", "county_name", "year_col", "month_col", "variable"] + day_cols

df = pd.read_csv(fpath, header=None, names=col_names, dtype={1: str})
print("Unique variable values:", df["variable"].unique())
print("\nFirst row values:")
print(df.iloc[0][day_cols].values)

Unique variable values: <ArrowStringArray>
['PRCP']
Length: 1, dtype: str

First row values:
[8.6 13.43 0.0 3.91 0.0 0.0 10.79 10.43 6.65 0.0 0.0 0.0 16.91 0.0 0.0
 2.93 0.0 0.0 0.0 0.0 2.26 7.2 0.0 8.05 0.95 16.29 0.83 0.0 11.46 15.15
 0.33]


In [4]:
import pandas as pd
import numpy as np
import os

# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project"
URDB_PATH  = os.path.join(OUTPUT_DIR, "OpenEI/usurdb.csv.gz")

# ── STEP 1: EXTRACT RESIDENTIAL FIXED CHARGES FROM URDB ──────────────────────
print("Loading URDB...")
cols = ["eiaid", "sector", "is_default", "startdate", "enddate",
        "fixedchargefirstmeter", "fixedchargeunits"]

urdb = pd.read_csv(URDB_PATH, compression="gzip", usecols=cols, low_memory=False)
print(f"Total rates: {len(urdb)}")

# Filter to residential sector only
urdb_res = urdb[urdb["sector"].str.lower().str.contains("residential", na=False)].copy()
print(f"Residential rates: {len(urdb_res)}")

# Keep only $/month units (drop $/day)
urdb_res = urdb_res[urdb_res["fixedchargeunits"] == "$/month"].copy()
print(f"Residential $/month rates: {len(urdb_res)}")

# Drop rows with missing fixed charge or missing eiaid
urdb_res = urdb_res.dropna(subset=["fixedchargefirstmeter", "eiaid"])
print(f"After dropping missing: {len(urdb_res)}")

# Convert eiaid to integer for join
urdb_res["eiaid"] = urdb_res["eiaid"].astype(int)

# Convert fixed charge to numeric
urdb_res["fixedchargefirstmeter"] = pd.to_numeric(
    urdb_res["fixedchargefirstmeter"], errors="coerce")

# Drop outliers — fixed charges above $100/month are commercial/industrial misclassified
urdb_res = urdb_res[urdb_res["fixedchargefirstmeter"] <= 100]
urdb_res = urdb_res[urdb_res["fixedchargefirstmeter"] >= 0]
print(f"After outlier removal: {len(urdb_res)}")

# Aggregate to utility level — mean fixed charge per utility
# Prefer default rates if available, otherwise use all residential rates
utility_charges = (urdb_res.groupby("eiaid")
                   .agg(
                       fixed_charge_mean = ("fixedchargefirstmeter", "mean"),
                       fixed_charge_median = ("fixedchargefirstmeter", "median"),
                       n_rates = ("fixedchargefirstmeter", "count"),
                       has_default = ("is_default", lambda x: x.eq(True).any())
                   )
                   .reset_index())

print(f"\nUtility-level fixed charges: {len(utility_charges)} utilities")
print(f"Summary stats:")
print(utility_charges["fixed_charge_mean"].describe().round(2))

# ── STEP 2: MERGE TO EIA SALES DATA ──────────────────────────────────────────
# Use annual EIA 861 sales to get utility-to-county mapping with customer counts
sales = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861_sales.csv"))
sales["utility_id"] = sales["utility_id"].astype(int)

# Merge fixed charges to sales on utility_id = eiaid
sales_fc = sales.merge(
    utility_charges[["eiaid", "fixed_charge_mean", "fixed_charge_median", "n_rates"]],
    left_on="utility_id", right_on="eiaid", how="left"
)

match_rate = sales_fc["fixed_charge_mean"].notna().mean()
print(f"\nUtility match rate to URDB: {match_rate:.1%}")
print(f"Utilities matched: {sales_fc['fixed_charge_mean'].notna().sum()}")
print(f"Utilities unmatched: {sales_fc['fixed_charge_mean'].isna().sum()}")

# ── STEP 3: AGGREGATE TO COUNTY LEVEL ────────────────────────────────────────
# Load territory file for utility-to-county mapping
territory = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861_territory.csv"))
county_fips = pd.read_csv(os.path.join(OUTPUT_DIR, "county_fips.csv"),
                          sep="|", dtype=str)

# Clean county names (same function as master merge)
def clean_county(name):
    if pd.isna(name):
        return name
    s = str(name)
    s = pd.Series([s]).str.replace(
        r"\s+(County|Parish|Borough|Municipality|Census Area|"
        r"City and Borough|Municipio|District|Island|city|City)$",
        "", regex=True).iloc[0]
    s = s.strip().lower()
    s = s.replace("st. ", "saint ").replace("st ", "saint ")
    s = s.replace(".", "")
    return s.strip()

county_fips.columns = [c.strip().lower() for c in county_fips.columns]
county_fips["FIPS"] = (county_fips["statefp"].str.zfill(2) +
                        county_fips["countyfp"].str.zfill(3))
county_fips["county_clean"] = county_fips["countyname"].apply(clean_county)
county_fips["state_abbr"]   = county_fips["state"].str.upper()

territory["county_clean"] = territory["county_name"].apply(clean_county)
territory["state_abbr"]   = territory["state"].str.upper()

territory_fips = territory.merge(
    county_fips[["state_abbr", "county_clean", "FIPS"]],
    on=["state_abbr", "county_clean"], how="left"
)

# Merge sales+fixed charges to territory
utility_county_fc = sales_fc.merge(
    territory_fips[["utility_id", "year", "state", "FIPS"]],
    on=["utility_id", "year"], how="inner"
)
utility_county_fc = utility_county_fc.dropna(subset=["FIPS"])

# Customer-weighted average fixed charge per county
# Weight by residential customers so larger utilities count more
def weighted_mean(group, values, weights):
    mask = group[values].notna() & group[weights].notna()
    if mask.sum() == 0:
        return np.nan
    return np.average(group.loc[mask, values], weights=group.loc[mask, weights])

county_fc = (utility_county_fc.groupby("FIPS")
             .apply(lambda g: pd.Series({
                 "fixed_charge_county": weighted_mean(
                     g, "fixed_charge_mean", "res_customers"),
                 "n_utilities_urdb": g["fixed_charge_mean"].notna().sum(),
                 "pct_utilities_matched": g["fixed_charge_mean"].notna().mean(),
             }))
             .reset_index())

county_fc["FIPS"] = county_fc["FIPS"].str.zfill(5)

print(f"\nCounty-level fixed charges: {len(county_fc)} counties")
print(f"Counties with fixed charge data: {county_fc['fixed_charge_county'].notna().sum()}")
print(f"\nFixed charge distribution:")
print(county_fc["fixed_charge_county"].describe().round(2))

# ── STEP 4: MERGE TO MASTER PANEL ────────────────────────────────────────────
# Fixed charges are time-invariant — merge on FIPS only
master = pd.read_csv(os.path.join(OUTPUT_DIR, "master_panel_monthly.csv"),
                     dtype={"FIPS": str})
master["FIPS"] = master["FIPS"].str.zfill(5)

master_fc = master.merge(
    county_fc[["FIPS", "fixed_charge_county", "pct_utilities_matched"]],
    on="FIPS", how="left"
)

# Demean fixed charge for interpretation
mean_fc = master_fc["fixed_charge_county"].mean()
master_fc["fixed_charge_demeaned"] = master_fc["fixed_charge_county"] - mean_fc

# Triple interaction: heat x poverty x fixed charge
for bin_label in ["bin7_20to25", "bin8_25to30", "bin9_over30"]:
    master_fc[f"{bin_label}_x_pov_x_fc"] = (
        master_fc[bin_label] *
        master_fc["poverty_demeaned"] *
        master_fc["fixed_charge_demeaned"]
    )

print(f"\nMaster panel with fixed charges: {len(master_fc)} rows")
print(f"Missing fixed charge: {master_fc['fixed_charge_county'].isna().sum()} "
      f"({master_fc['fixed_charge_county'].isna().mean():.1%})")
print(f"\nSample — Autauga County:")
print(master_fc[master_fc["FIPS"] == "01001"][
    ["FIPS", "year", "month", "fixed_charge_county",
     "fixed_charge_demeaned"]
].head(3).to_string())

# ── SAVE ──────────────────────────────────────────────────────────────────────
master_fc.to_csv(os.path.join(OUTPUT_DIR, "master_panel_monthly_fc.csv"),
                 index=False)
county_fc.to_csv(os.path.join(OUTPUT_DIR, "county_fixed_charges.csv"),
                 index=False)

print(f"\nSaved master_panel_monthly_fc.csv")
print(f"Saved county_fixed_charges.csv")
print(f"Mean fixed charge across counties: ${mean_fc:.2f}/month")

Loading URDB...
Total rates: 58275
Residential rates: 13620
Residential $/month rates: 12560
After dropping missing: 12044
After outlier removal: 11972

Utility-level fixed charges: 2451 utilities
Summary stats:
count    2451.00
mean       13.67
std         8.78
min         0.00
25%         7.67
50%        11.50
75%        17.50
max        74.00
Name: fixed_charge_mean, dtype: float64

Utility match rate to URDB: 59.5%
Utilities matched: 17805
Utilities unmatched: 12140

County-level fixed charges: 3115 counties
Counties with fixed charge data: 3104

Fixed charge distribution:
count    3104.00
mean       13.40
std         5.87
min         1.29
25%         9.76
50%        12.26
75%        15.48
max        69.11
Name: fixed_charge_county, dtype: float64

Master panel with fixed charges: 335878 rows
Missing fixed charge: 480 (0.1%)

Sample — Autauga County:
    FIPS  year  month  fixed_charge_county  fixed_charge_demeaned
0  01001  2015      1            10.076729              -2.486643
1

In [6]:
import pandas as pd
import numpy as np
import os

# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project"
URDB_PATH  = os.path.join(OUTPUT_DIR, "OpenEI/usurdb.csv.gz")

BIN_LABELS = ["bin1_under_neg5", "bin2_neg5to0", "bin3_0to5",
              "bin4_5to10",      "bin5_10to15",  "bin6_15to20",
              "bin7_20to25",     "bin8_25to30",  "bin9_over30"]
HOT_BINS   = ["bin7_20to25", "bin8_25to30", "bin9_over30"]

# ── LOAD ALL DATA ─────────────────────────────────────────────────────────────
print("Loading data...")
sales_m   = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861m_sales.csv"))
territory = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861_territory.csv"))
fips_ref  = pd.read_csv(os.path.join(OUTPUT_DIR, "county_fips.csv"), sep="|", dtype=str)
noaa      = pd.read_csv(os.path.join(OUTPUT_DIR, "noaa_panel_final.csv"), dtype={"FIPS": str})
acs_pov   = pd.read_csv(os.path.join(OUTPUT_DIR, "acs_poverty_panel.csv"), dtype={"FIPS": str})
acs_ctrl  = pd.read_csv(os.path.join(OUTPUT_DIR, "acs_controls_panel.csv"), dtype={"FIPS": str})

sales_m["year"]   = sales_m["year"].astype(int)
sales_m["month"]  = sales_m["month"].astype(int)
territory["year"] = territory["year"].astype(int)
acs_ctrl["year"]  = acs_ctrl["year"].astype(int)

print(f"Monthly sales: {len(sales_m):,} | Territory: {len(territory):,} | "
      f"NOAA: {len(noaa):,} | ACS poverty: {len(acs_pov):,} | ACS controls: {len(acs_ctrl):,}")

# ── COUNTY NAME CLEANER ───────────────────────────────────────────────────────
def clean_county(name):
    if pd.isna(name): return name
    s = pd.Series([str(name)]).str.replace(
        r"\s+(County|Parish|Borough|Municipality|Census Area|"
        r"City and Borough|Municipio|District|Island|city|City)$",
        "", regex=True).iloc[0]
    s = s.strip().lower()
    s = s.replace("st. ", "saint ").replace("st ", "saint ").replace(".", "")
    return s.strip()

# ── STEP 1: FIPS CROSSWALK ────────────────────────────────────────────────────
fips_ref.columns         = [c.strip().lower() for c in fips_ref.columns]
fips_ref["FIPS"]         = fips_ref["statefp"].str.zfill(2) + fips_ref["countyfp"].str.zfill(3)
fips_ref["county_clean"] = fips_ref["countyname"].apply(clean_county)
fips_ref["state_abbr"]   = fips_ref["state"].str.upper()

territory["county_clean"] = territory["county_name"].apply(clean_county)
territory["state_abbr"]   = territory["state"].str.upper()

territory_fips = territory.merge(
    fips_ref[["state_abbr", "county_clean", "FIPS"]],
    on=["state_abbr", "county_clean"], how="left"
)
print(f"Territory FIPS match rate: {territory_fips['FIPS'].notna().mean():.1%}")

# ── STEP 2: MERGE MONTHLY SALES TO TERRITORY ─────────────────────────────────
utility_county_m = sales_m.merge(
    territory_fips[["utility_id", "year", "state", "FIPS"]],
    on=["utility_id", "year", "state"], how="inner"
)

# ── STEP 3: AGGREGATE TO COUNTY-MONTH ────────────────────────────────────────
county_month = (utility_county_m.dropna(subset=["FIPS"])
                .groupby(["FIPS", "year", "month"])
                .agg(
                    res_revenue_000_total = ("res_revenue_000", "sum"),
                    res_sales_mwh_total   = ("res_sales_mwh",   "sum"),
                    res_customers_total   = ("res_customers",   "sum"),
                    n_utilities           = ("utility_id",      "nunique"),
                ).reset_index())

county_month["implied_price_per_mwh"] = np.where(
    (county_month["res_revenue_000_total"] > 0) & (county_month["res_sales_mwh_total"] > 0),
    (county_month["res_revenue_000_total"] * 1000) / county_month["res_sales_mwh_total"],
    np.nan)

county_month["res_sales_per_customer"] = np.where(
    county_month["res_customers_total"] > 0,
    county_month["res_sales_mwh_total"] / county_month["res_customers_total"],
    np.nan)

county_month["log_res_sales_per_customer"] = np.log(
    county_month["res_sales_per_customer"].replace(0, np.nan))
county_month["log_implied_price"] = np.log(
    county_month["implied_price_per_mwh"].replace(0, np.nan))

ownership_m = (utility_county_m.dropna(subset=["FIPS", "res_customers"])
               .sort_values("res_customers", ascending=False)
               .groupby(["FIPS", "year", "month"])
               .agg(dominant_ownership_type=("ownership_type", "first"))
               .reset_index())
county_month = county_month.merge(ownership_m, on=["FIPS", "year", "month"], how="left")

print(f"County-month panel: {len(county_month):,} rows | {county_month['FIPS'].nunique():,} counties")

# ── STEP 4: MERGE NOAA ────────────────────────────────────────────────────────
for df in [noaa, acs_pov, acs_ctrl]:
    df["FIPS"] = df["FIPS"].str.zfill(5)
county_month["FIPS"] = county_month["FIPS"].str.zfill(5)

noaa_cols = ["FIPS", "year", "month"] + BIN_LABELS + ["prcp_monthly_mm"]
master = county_month.merge(noaa[noaa_cols], on=["FIPS", "year", "month"], how="left")

# Fix precipitation sentinel values
master["prcp_monthly_mm"] = master["prcp_monthly_mm"].where(
    master["prcp_monthly_mm"] >= 0, np.nan)
master["prcp_monthly_mm_sq"] = master["prcp_monthly_mm"] ** 2

# ── STEP 5: MERGE ACS POVERTY ─────────────────────────────────────────────────
master = master.merge(
    acs_pov[["FIPS", "year", "pct_below_150pct_poverty", "total_pop"]],
    on=["FIPS", "year"], how="left")

# ── STEP 6: MERGE ACS CONTROLS ────────────────────────────────────────────────
master = master.merge(
    acs_ctrl[["FIPS", "year", "median_hh_income", "pct_white", "pct_black",
               "pct_hispanic", "pct_bachelors_plus", "unemployment_rate"]],
    on=["FIPS", "year"], how="left")

# ── STEP 7: MERGE FIXED CHARGES ───────────────────────────────────────────────
print("\nLoading URDB...")
urdb = pd.read_csv(URDB_PATH, compression="gzip",
                   usecols=["eiaid", "sector", "fixedchargefirstmeter", "fixedchargeunits"],
                   low_memory=False)
urdb_res = urdb[urdb["sector"].str.lower().str.contains("residential", na=False)].copy()
urdb_res = urdb_res[urdb_res["fixedchargeunits"] == "$/month"].copy()
urdb_res = urdb_res.dropna(subset=["fixedchargefirstmeter", "eiaid"])
urdb_res["eiaid"] = urdb_res["eiaid"].astype(int)
urdb_res["fixedchargefirstmeter"] = pd.to_numeric(
    urdb_res["fixedchargefirstmeter"], errors="coerce")
urdb_res = urdb_res[(urdb_res["fixedchargefirstmeter"] >= 0) &
                    (urdb_res["fixedchargefirstmeter"] <= 100)]

utility_charges = (urdb_res.groupby("eiaid")
                   .agg(fixed_charge_mean=("fixedchargefirstmeter", "mean"))
                   .reset_index())

sales_eia = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861_sales.csv"))
sales_eia["utility_id"] = sales_eia["utility_id"].astype(int)
sales_fc = sales_eia.merge(utility_charges, left_on="utility_id", right_on="eiaid", how="left")

territory_fips2 = territory.merge(
    fips_ref[["state_abbr", "county_clean", "FIPS"]],
    on=["state_abbr", "county_clean"], how="left")
utility_county_fc = sales_fc.merge(
    territory_fips2[["utility_id", "year", "state", "FIPS"]],
    on=["utility_id", "year"], how="inner").dropna(subset=["FIPS"])

utility_county_fc["fc_x_customers"] = (utility_county_fc["fixed_charge_mean"] *
                                        utility_county_fc["res_customers"])
county_fc = (utility_county_fc.groupby("FIPS")
             .agg(fc_x_customers_sum=("fc_x_customers", "sum"),
                  customers_sum=("res_customers", "sum"))
             .reset_index())
county_fc["fixed_charge_county"] = (county_fc["fc_x_customers_sum"] /
                                     county_fc["customers_sum"])
county_fc["FIPS"] = county_fc["FIPS"].str.zfill(5)

master = master.merge(county_fc[["FIPS", "fixed_charge_county"]], on="FIPS", how="left")

# ── STEP 8: CONSTRUCT REGRESSION VARIABLES ───────────────────────────────────
mean_pov = master["pct_below_150pct_poverty"].mean()
mean_fc  = master["fixed_charge_county"].mean()

master["poverty_demeaned"] = master["pct_below_150pct_poverty"] - mean_pov
master["fc_demeaned"]      = master["fixed_charge_county"] - mean_fc

master["prcp_monthly_mm_sq"] = master["prcp_monthly_mm"] ** 2

# Heat x poverty interactions — all bins
for b in BIN_LABELS:
    if b != "bin6_15to20":
        master[f"{b}_x_pov"] = master[b] * master["poverty_demeaned"]

# Fixed charge triple interaction
for b in HOT_BINS:
    master[f"{b}_x_pov_x_fc"] = master[b] * master["poverty_demeaned"] * master["fc_demeaned"]

# Ownership dummies and triple interactions
master["is_coop"]      = (master["dominant_ownership_type"] == "Cooperative").astype(int)
master["is_municipal"] = (master["dominant_ownership_type"] == "Municipal").astype(int)
for b in HOT_BINS:
    master[f"{b}_x_pov_x_coop"] = master[b] * master["poverty_demeaned"] * master["is_coop"]
    master[f"{b}_x_pov_x_muni"] = master[b] * master["poverty_demeaned"] * master["is_municipal"]

master["county_month_fe"] = master["FIPS"] + "_" + master["month"].astype(str).str.zfill(2)

# ── DIAGNOSTICS ───────────────────────────────────────────────────────────────
print(f"\nFinal master panel: {len(master):,} rows | "
      f"{master['FIPS'].nunique():,} counties | "
      f"Years: {master['year'].min()}--{master['year'].max()}")

print(f"\nMissing values:")
check_vars = ["log_res_sales_per_customer", "pct_below_150pct_poverty",
              "prcp_monthly_mm", "bin9_over30", "fixed_charge_county",
              "median_hh_income", "pct_white", "unemployment_rate"]
for v in check_vars:
    n = master[v].isna().sum()
    print(f"  {v:40s}: {n:6,} ({n/len(master):.1%})")

print(f"\nMean poverty: {mean_pov:.3f} | Mean fixed charge: ${mean_fc:.2f}/mo")

print(f"\nAutauga County (01001) -- 2020 summer:")
print(master[(master["FIPS"]=="01001") & (master["year"]==2020) &
             (master["month"].isin([6,7,8]))][
    ["FIPS","year","month","bin8_25to30","prcp_monthly_mm",
     "pct_below_150pct_poverty","median_hh_income","fixed_charge_county"]
].to_string())

# ── SAVE ──────────────────────────────────────────────────────────────────────
master.to_csv(os.path.join(OUTPUT_DIR, "master_panel_monthly_fc.csv"), index=False)
print(f"\nSaved master_panel_monthly_fc.csv")

Loading data...
Monthly sales: 48,455 | Territory: 118,147 | NOAA: 372,840 | ACS poverty: 41,869 | ACS controls: 32,208
Territory FIPS match rate: 99.5%
County-month panel: 335,878 rows | 2,814 counties

Loading URDB...

Final master panel: 335,878 rows | 2,814 counties | Years: 2015--2024

Missing values:
  log_res_sales_per_customer              :      3 (0.0%)
  pct_below_150pct_poverty                :    300 (0.1%)
  prcp_monthly_mm                         :  2,412 (0.7%)
  bin9_over30                             :  2,412 (0.7%)
  fixed_charge_county                     :      0 (0.0%)
  median_hh_income                        :    336 (0.1%)
  pct_white                               :    288 (0.1%)
  unemployment_rate                       :    300 (0.1%)

Mean poverty: 0.251 | Mean fixed charge: $11.86/mo

Autauga County (01001) -- 2020 summer:
     FIPS  year  month  bin8_25to30  prcp_monthly_mm  pct_below_150pct_poverty  median_hh_income  fixed_charge_county
65  01001  2020   

In [13]:
import pandas as pd

fc    = pd.read_csv("/Users/aliabonanno/public_policy_project/county_fixed_charges.csv")
master = pd.read_csv("/Users/aliabonanno/public_policy_project/master_panel_monthly_fc.csv",
                     dtype={"FIPS": str})

fc["FIPS"] = fc["FIPS"].astype(str).str.zfill(5)
master["FIPS"] = master["FIPS"].str.zfill(5)

# Get dominant ownership per county (take mode across months)
ownership = (master.groupby("FIPS")["dominant_ownership_type"]
             .agg(lambda x: x.mode()[0])
             .reset_index())

merged = fc.merge(ownership, on="FIPS", how="left")

result = (merged.groupby("dominant_ownership_type")
          .agg(n_counties=("FIPS", "count"),
               mean_match_rate=("pct_utilities_matched", "mean"),
               mean_fixed_charge=("fixed_charge_county", "mean"))
          .round(3))

print(result)

                         n_counties  mean_match_rate  mean_fixed_charge
dominant_ownership_type                                                
Cooperative                     163            0.946             17.847
IOU                            2534            0.922             11.962
Municipal                       117            0.891             18.703


In [19]:
import pandas as pd

sales = pd.read_csv("/Users/aliabonanno/public_policy_project/eia861_sales.csv")
urdb  = pd.read_csv("/Users/aliabonanno/public_policy_project/OpenEI/usurdb.csv.gz",
                    compression="gzip",
                    usecols=["eiaid", "sector", "fixedchargefirstmeter", "fixedchargeunits"],
                    low_memory=False)

urdb_res = urdb[urdb["sector"].str.lower().str.contains("residential", na=False)].copy()
urdb_res = urdb_res[urdb_res["fixedchargeunits"] == "$/month"].dropna(subset=["eiaid"])
urdb_res["eiaid"] = urdb_res["eiaid"].astype(int)
matched_ids = set(urdb_res["eiaid"].unique())

sales["utility_id"] = sales["utility_id"].astype(int)
sales["matched"] = sales["utility_id"].isin(matched_ids)

# Deduplicate to one row per utility
utilities = sales.drop_duplicates(subset=["utility_id"])

result = (utilities.groupby("ownership")
          .agg(total=("utility_id", "count"),
               matched=("matched", "sum"))
          .assign(match_rate=lambda x: (x["matched"] / x["total"]).round(3)))

print(result)
print(f"\nOverall: {utilities['matched'].mean():.1%}")
print(result)
print(f"\nOverall: {utilities['matched'].mean():.1%}")

                             total  matched  match_rate
ownership                                              
Behind the Meter                15        0       0.000
Community Choice Aggregator     26        1       0.038
Cooperative                    815      671       0.823
Federal                          6        2       0.333
Investor Owned                 164      147       0.896
Municipal                     1773     1499       0.845
Political Subdivision           92       68       0.739
Retail Power Marketer          351        2       0.006
State                           10        6       0.600

Overall: 73.7%
                             total  matched  match_rate
ownership                                              
Behind the Meter                15        0       0.000
Community Choice Aggregator     26        1       0.038
Cooperative                    815      671       0.823
Federal                          6        2       0.333
Investor Owned                 1

In [27]:
rucc = pd.read_csv("/Users/aliabonanno/public_policy_project/rucc_codes.csv",
                   encoding="latin-1")
print(rucc.columns.tolist())
print(rucc.head(3))
print(f"Shape: {rucc.shape}")

['FIPS', 'State', 'County_Name', 'Attribute', 'Value']
   FIPS State     County_Name        Attribute  \
0  1001    AL  Autauga County  Population_2020   
1  1001    AL  Autauga County        RUCC_2023   
2  1001    AL  Autauga County      Description   

                                               Value  
0                                              58805  
1                                                  2  
2  Metro - Counties in metro areas of 250,000 to ...  
Shape: (9703, 5)


In [29]:
import pandas as pd
import numpy as np
import os

# ── CONFIG ───────────────────────────────────────────────────────────────────
OUTPUT_DIR = "/Users/aliabonanno/public_policy_project"
URDB_PATH  = os.path.join(OUTPUT_DIR, "OpenEI/usurdb.csv.gz")

BIN_LABELS = ["bin1_under_neg5", "bin2_neg5to0", "bin3_0to5",
              "bin4_5to10",      "bin5_10to15",  "bin6_15to20",
              "bin7_20to25",     "bin8_25to30",  "bin9_over30"]
HOT_BINS   = ["bin7_20to25", "bin8_25to30", "bin9_over30"]

# ── LOAD ALL DATA ─────────────────────────────────────────────────────────────
print("Loading data...")
sales_m   = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861m_sales.csv"))
territory = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861_territory.csv"))
fips_ref  = pd.read_csv(os.path.join(OUTPUT_DIR, "county_fips.csv"), sep="|", dtype=str)
noaa      = pd.read_csv(os.path.join(OUTPUT_DIR, "noaa_panel_final.csv"), dtype={"FIPS": str})
acs_pov   = pd.read_csv(os.path.join(OUTPUT_DIR, "acs_poverty_panel.csv"), dtype={"FIPS": str})
acs_ctrl  = pd.read_csv(os.path.join(OUTPUT_DIR, "acs_controls_panel.csv"), dtype={"FIPS": str})

# ── LOAD AND CLEAN RUCC ───────────────────────────────────────────────────────
print("Loading RUCC codes...")
rucc_raw = pd.read_csv(os.path.join(OUTPUT_DIR, "rucc_codes.csv"), encoding="latin-1")

# Pivot from long to wide — keep only RUCC_2023 row
rucc = (rucc_raw[rucc_raw["Attribute"] == "RUCC_2023"]
        [["FIPS", "Value"]]
        .rename(columns={"Value": "rucc_score"}))
rucc["FIPS"]       = rucc["FIPS"].astype(str).str.zfill(5)
rucc["rucc_score"] = pd.to_numeric(rucc["rucc_score"], errors="coerce")

# Rural flag: RUCC >= 4 is nonmetro (rural)
rucc["rural"] = (rucc["rucc_score"] >= 4).astype(int)

print(f"RUCC counties: {len(rucc):,}")
print(f"RUCC distribution:\n{rucc['rucc_score'].value_counts().sort_index()}")

# Fix types
sales_m["year"]   = sales_m["year"].astype(int)
sales_m["month"]  = sales_m["month"].astype(int)
territory["year"] = territory["year"].astype(int)
acs_ctrl["year"]  = acs_ctrl["year"].astype(int)

print(f"Monthly sales: {len(sales_m):,} | Territory: {len(territory):,} | "
      f"NOAA: {len(noaa):,} | ACS poverty: {len(acs_pov):,} | "
      f"ACS controls: {len(acs_ctrl):,}")

# ── COUNTY NAME CLEANER ───────────────────────────────────────────────────────
def clean_county(name):
    if pd.isna(name): return name
    s = pd.Series([str(name)]).str.replace(
        r"\s+(County|Parish|Borough|Municipality|Census Area|"
        r"City and Borough|Municipio|District|Island|city|City)$",
        "", regex=True).iloc[0]
    s = s.strip().lower()
    s = s.replace("st. ", "saint ").replace("st ", "saint ").replace(".", "")
    return s.strip()

# ── STEP 1: FIPS CROSSWALK ────────────────────────────────────────────────────
fips_ref.columns         = [c.strip().lower() for c in fips_ref.columns]
fips_ref["FIPS"]         = fips_ref["statefp"].str.zfill(2) + fips_ref["countyfp"].str.zfill(3)
fips_ref["county_clean"] = fips_ref["countyname"].apply(clean_county)
fips_ref["state_abbr"]   = fips_ref["state"].str.upper()

territory["county_clean"] = territory["county_name"].apply(clean_county)
territory["state_abbr"]   = territory["state"].str.upper()

territory_fips = territory.merge(
    fips_ref[["state_abbr", "county_clean", "FIPS"]],
    on=["state_abbr", "county_clean"], how="left"
)
print(f"Territory FIPS match rate: {territory_fips['FIPS'].notna().mean():.1%}")

# ── STEP 2: MERGE MONTHLY SALES TO TERRITORY ─────────────────────────────────
utility_county_m = sales_m.merge(
    territory_fips[["utility_id", "year", "state", "FIPS"]],
    on=["utility_id", "year", "state"], how="inner"
)

# ── STEP 3: AGGREGATE TO COUNTY-MONTH ────────────────────────────────────────
county_month = (utility_county_m.dropna(subset=["FIPS"])
                .groupby(["FIPS", "year", "month"])
                .agg(
                    res_revenue_000_total = ("res_revenue_000", "sum"),
                    res_sales_mwh_total   = ("res_sales_mwh",   "sum"),
                    res_customers_total   = ("res_customers",   "sum"),
                    n_utilities           = ("utility_id",      "nunique"),
                ).reset_index())

county_month["implied_price_per_mwh"] = np.where(
    (county_month["res_revenue_000_total"] > 0) &
    (county_month["res_sales_mwh_total"] > 0),
    (county_month["res_revenue_000_total"] * 1000) /
     county_month["res_sales_mwh_total"], np.nan)

county_month["res_sales_per_customer"] = np.where(
    county_month["res_customers_total"] > 0,
    county_month["res_sales_mwh_total"] / county_month["res_customers_total"],
    np.nan)

county_month["log_res_sales_per_customer"] = np.log(
    county_month["res_sales_per_customer"].replace(0, np.nan))
county_month["log_implied_price"] = np.log(
    county_month["implied_price_per_mwh"].replace(0, np.nan))

ownership_m = (utility_county_m.dropna(subset=["FIPS", "res_customers"])
               .sort_values("res_customers", ascending=False)
               .groupby(["FIPS", "year", "month"])
               .agg(dominant_ownership_type=("ownership_type", "first"))
               .reset_index())
county_month = county_month.merge(
    ownership_m, on=["FIPS", "year", "month"], how="left")

print(f"County-month panel: {len(county_month):,} rows | "
      f"{county_month['FIPS'].nunique():,} counties")

# ── STEP 4: MERGE NOAA ────────────────────────────────────────────────────────
for df in [noaa, acs_pov, acs_ctrl]:
    df["FIPS"] = df["FIPS"].str.zfill(5)
county_month["FIPS"] = county_month["FIPS"].str.zfill(5)

noaa_cols = ["FIPS", "year", "month"] + BIN_LABELS + ["prcp_monthly_mm"]
master = county_month.merge(noaa[noaa_cols], on=["FIPS", "year", "month"], how="left")

# Fix precipitation sentinel values
master["prcp_monthly_mm"] = master["prcp_monthly_mm"].where(
    master["prcp_monthly_mm"] >= 0, np.nan)
master["prcp_monthly_mm_sq"] = master["prcp_monthly_mm"] ** 2

# ── STEP 5: MERGE ACS POVERTY ─────────────────────────────────────────────────
master = master.merge(
    acs_pov[["FIPS", "year", "pct_below_150pct_poverty", "total_pop"]],
    on=["FIPS", "year"], how="left")

# ── STEP 6: MERGE ACS CONTROLS (including electric heating) ──────────────────
master = master.merge(
    acs_ctrl[["FIPS", "year", "median_hh_income", "pct_white", "pct_black",
               "pct_hispanic", "pct_bachelors_plus", "unemployment_rate",
               "pct_electric_heat"]],
    on=["FIPS", "year"], how="left")

# ── STEP 7: MERGE RUCC ────────────────────────────────────────────────────────
master = master.merge(rucc[["FIPS", "rucc_score", "rural"]], on="FIPS", how="left")
print(f"RUCC match rate: {master['rucc_score'].notna().mean():.1%}")

# ── STEP 8: MERGE FIXED CHARGES ───────────────────────────────────────────────
print("\nLoading URDB...")
urdb = pd.read_csv(URDB_PATH, compression="gzip",
                   usecols=["eiaid", "sector", "fixedchargefirstmeter",
                             "fixedchargeunits"], low_memory=False)
urdb_res = urdb[urdb["sector"].str.lower().str.contains("residential", na=False)].copy()
urdb_res = urdb_res[urdb_res["fixedchargeunits"] == "$/month"].copy()
urdb_res = urdb_res.dropna(subset=["fixedchargefirstmeter", "eiaid"])
urdb_res["eiaid"] = urdb_res["eiaid"].astype(int)
urdb_res["fixedchargefirstmeter"] = pd.to_numeric(
    urdb_res["fixedchargefirstmeter"], errors="coerce")
urdb_res = urdb_res[(urdb_res["fixedchargefirstmeter"] >= 0) &
                    (urdb_res["fixedchargefirstmeter"] <= 100)]

utility_charges = (urdb_res.groupby("eiaid")
                   .agg(fixed_charge_mean=("fixedchargefirstmeter", "mean"))
                   .reset_index())

sales_eia = pd.read_csv(os.path.join(OUTPUT_DIR, "eia861_sales.csv"))
sales_eia["utility_id"] = sales_eia["utility_id"].astype(int)
sales_fc = sales_eia.merge(
    utility_charges, left_on="utility_id", right_on="eiaid", how="left")

territory_fips2 = territory.merge(
    fips_ref[["state_abbr", "county_clean", "FIPS"]],
    on=["state_abbr", "county_clean"], how="left")
utility_county_fc = sales_fc.merge(
    territory_fips2[["utility_id", "year", "state", "FIPS"]],
    on=["utility_id", "year"], how="inner").dropna(subset=["FIPS"])

utility_county_fc["fc_x_customers"] = (utility_county_fc["fixed_charge_mean"] *
                                        utility_county_fc["res_customers"])
county_fc = (utility_county_fc.groupby("FIPS")
             .agg(fc_x_customers_sum=("fc_x_customers", "sum"),
                  customers_sum=("res_customers", "sum"))
             .reset_index())
county_fc["fixed_charge_county"] = (county_fc["fc_x_customers_sum"] /
                                     county_fc["customers_sum"])
county_fc["FIPS"] = county_fc["FIPS"].str.zfill(5)

master = master.merge(
    county_fc[["FIPS", "fixed_charge_county"]], on="FIPS", how="left")

# ── STEP 9: CONSTRUCT REGRESSION VARIABLES ───────────────────────────────────
mean_pov = master["pct_below_150pct_poverty"].mean()
mean_fc  = master["fixed_charge_county"].mean()

master["poverty_demeaned"] = master["pct_below_150pct_poverty"] - mean_pov
master["fc_demeaned"]      = master["fixed_charge_county"] - mean_fc
master["prcp_monthly_mm_sq"] = master["prcp_monthly_mm"] ** 2

# COVID indicator
master["covid"] = (master["year"] == 2020).astype(int)

# Heat x poverty interactions — all bins
for b in BIN_LABELS:
    if b != "bin6_15to20":
        master[f"{b}_x_pov"] = master[b] * master["poverty_demeaned"]

# Fixed charge triple interaction
for b in HOT_BINS:
    master[f"{b}_x_pov_x_fc"] = (master[b] * master["poverty_demeaned"] *
                                   master["fc_demeaned"])

# Ownership dummies and triple interactions
master["is_coop"]      = (master["dominant_ownership_type"] == "Cooperative").astype(int)
master["is_municipal"] = (master["dominant_ownership_type"] == "Municipal").astype(int)
for b in HOT_BINS:
    master[f"{b}_x_pov_x_coop"] = (master[b] * master["poverty_demeaned"] *
                                    master["is_coop"])
    master[f"{b}_x_pov_x_muni"] = (master[b] * master["poverty_demeaned"] *
                                    master["is_municipal"])

# Rural x poverty x bin interactions (for rurality robustness)
for b in HOT_BINS:
    master[f"{b}_x_pov_x_rural"] = (master[b] * master["poverty_demeaned"] *
                                     master["rural"])

# COVID x bin interactions (for COVID robustness)
for b in HOT_BINS:
    master[f"{b}_x_covid"] = master[b] * master["covid"]

master["county_month_fe"] = (master["FIPS"] + "_" +
                              master["month"].astype(str).str.zfill(2))

# ── DIAGNOSTICS ───────────────────────────────────────────────────────────────
print(f"\nFinal master panel: {len(master):,} rows | "
      f"{master['FIPS'].nunique():,} counties | "
      f"Years: {master['year'].min()}--{master['year'].max()}")

print(f"\nMissing values:")
check_vars = ["log_res_sales_per_customer", "pct_below_150pct_poverty",
              "prcp_monthly_mm", "bin9_over30", "fixed_charge_county",
              "median_hh_income", "pct_electric_heat",
              "rucc_score", "unemployment_rate"]
for v in check_vars:
    n = master[v].isna().sum()
    print(f"  {v:40s}: {n:6,} ({n/len(master):.1%})")

print(f"\nMean poverty: {mean_pov:.3f} | Mean fixed charge: ${mean_fc:.2f}/mo")
print(f"Mean electric heat share: {master['pct_electric_heat'].mean():.3f}")
print(f"Mean RUCC score: {master['rucc_score'].mean():.2f}")
print(f"Rural counties (RUCC>=4): {master['rural'].mean():.1%}")

# Sanity check
print(f"\nAutauga County (01001) -- 2020 summer:")
print(master[(master["FIPS"]=="01001") & (master["year"]==2020) &
             (master["month"].isin([6,7,8]))][
    ["FIPS","year","month","bin8_25to30","pct_below_150pct_poverty",
     "pct_electric_heat","rucc_score","fixed_charge_county"]
].to_string())

# ── SAVE ──────────────────────────────────────────────────────────────────────
master.to_csv(os.path.join(OUTPUT_DIR, "master_panel_monthly_fc.csv"), index=False)
print(f"\nSaved master_panel_monthly_fc.csv")

Loading data...
Loading RUCC codes...
RUCC counties: 3,233
RUCC distribution:
rucc_score
1    483
2    398
3    371
4    204
5     81
6    385
7    248
8    468
9    595
Name: count, dtype: int64
Monthly sales: 48,455 | Territory: 118,147 | NOAA: 372,840 | ACS poverty: 41,869 | ACS controls: 32,208
Territory FIPS match rate: 99.5%
County-month panel: 335,878 rows | 2,814 counties
RUCC match rate: 99.7%

Loading URDB...

Final master panel: 335,878 rows | 2,814 counties | Years: 2015--2024

Missing values:
  log_res_sales_per_customer              :      3 (0.0%)
  pct_below_150pct_poverty                :    300 (0.1%)
  prcp_monthly_mm                         :  2,412 (0.7%)
  bin9_over30                             :  2,412 (0.7%)
  fixed_charge_county                     :      0 (0.0%)
  median_hh_income                        :    336 (0.1%)
  pct_electric_heat                       :    288 (0.1%)
  rucc_score                              :    960 (0.3%)
  unemployment_rate      

In [31]:
print(f"RUCC >= 4: {(master['rucc_score'] >= 4).mean():.1%}")
print(f"RUCC >= 7: {(master['rucc_score'] >= 7).mean():.1%}")
print(f"RUCC == 9: {(master['rucc_score'] == 9).mean():.1%}")

RUCC >= 4: 60.2%
RUCC >= 7: 38.9%
RUCC == 9: 16.5%


In [35]:
import pandas as pd

master = pd.read_csv("/Users/aliabonanno/public_policy_project/master_panel_monthly_fc.csv",
                     dtype={"FIPS": str})

counties = master.drop_duplicates(subset=["FIPS"])

for cutoff in [4, 7, 9]:
    counties[f"rural_{cutoff}"] = (counties["rucc_score"] >= cutoff).astype(int)
    print(f"\nRUCC >= {cutoff} rural cutoff:")
    print(counties.groupby("dominant_ownership_type")[f"rural_{cutoff}"].agg(
        n_counties="count",
        n_rural="sum",
        pct_rural="mean"
    ).round(3))
    
    # Cross-tab
    print(f"\nCross-tab ownership × rural (RUCC>={cutoff}):")
    print(pd.crosstab(
        counties["dominant_ownership_type"],
        counties[f"rural_{cutoff}"],
        margins=True,
        normalize="index"
    ).round(3))


RUCC >= 4 rural cutoff:
                         n_counties  n_rural  pct_rural
dominant_ownership_type                                
Cooperative                     165      130      0.788
IOU                            2532     1498      0.592
Municipal                       117       67      0.573

Cross-tab ownership × rural (RUCC>=4):
rural_4                      0      1
dominant_ownership_type              
Cooperative              0.212  0.788
IOU                      0.408  0.592
Municipal                0.427  0.573
All                      0.398  0.602

RUCC >= 7 rural cutoff:
                         n_counties  n_rural  pct_rural
dominant_ownership_type                                
Cooperative                     165      101      0.612
IOU                            2532      947      0.374
Municipal                       117       48      0.410

Cross-tab ownership × rural (RUCC>=7):
rural_7                      0      1
dominant_ownership_type              
Cooper